In [ ]:
!apt-get update -qq
!apt-get install -y -qq nginx

!pip install -q virtualenv
# Cloudflare tunnel
!curl -L -o cloudflared.deb \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared.deb

print("✅ Installation complete!")

In [ ]:
!python -m virtualenv /kaggle/working/venv

In [ ]:
!/kaggle/working/venv/bin/python --version

In [ ]:
!/kaggle/working/venv/bin/python -m pip install wrapt

In [ ]:
!/kaggle/working/venv/bin/python -m pip install -U pip setuptools wheel

In [ ]:
!/kaggle/working/venv/bin/python -m pip install -q \
    git+https://github.com/k2-fsa/OmniVoice.git \
    faster-whisper \
    fastapi \
    uvicorn \
    "uvicorn[standard]" \
    'omnivoice[tn]'

In [ ]:
!/kaggle/working/venv/bin/python -c "import numpy as np; import faster_whisper; import omnivoice; print('✅ faster-whisper imported successfully'); print(faster_whisper.__file__); print('✅ omnivoice imported successfully'); print(omnivoice.__file__)"

In [ ]:
!/kaggle/working/venv/bin/python -c "import asyncio; import json; import time; import io; import re; import numpy as np; import torch; import av; from fastapi import ( FastAPI, File, UploadFile, WebSocket, WebSocketDisconnect,); from faster_whisper import WhisperModel; from omnivoice import OmniVoice, VoiceClonePrompt"

In [ ]:
!/kaggle/working/venv/bin/python -c "import torch; dtype=getattr(torch, 'float16', None); print(f'✅ {dtype}');"

In [ ]:
%%writefile /kaggle/working/stt_server.py

import asyncio
import json
import time
import io
import re

import numpy as np
import torch
import av


from fastapi import (
    FastAPI,
    File,
    UploadFile,
    WebSocket,
    WebSocketDisconnect,
)

from faster_whisper import WhisperModel

MODEL_NAME = "large-v3-turbo"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

COMPUTE_TYPE = (
    "float16"
    if DEVICE == "cuda"
    else "int8"
)

SAMPLE_RATE = 16000

ALLOWED_LANGUAGES = {"en", "ar"}
ARABIC_UNICODE_RANGE = ("\u0600", "\u06FF")

# ============================================================
# LOAD MODEL
# ============================================================

print("=" * 60)
print("FASTER-WHISPER STT SERVER")
print("=" * 60)

print("Model       :", MODEL_NAME)
print("Device      :", DEVICE)
print("Compute     :", COMPUTE_TYPE)
print("Sample rate :", SAMPLE_RATE)


print()
print("Loading Faster-Whisper...")

whisper_model = WhisperModel(
    MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    num_workers=1,
)

print("✅ Faster-Whisper loaded")

# ============================================================
# FASTAPI
# ============================================================

app = FastAPI(
    title="Kaggle Faster-Whisper STT"
)

@app.get("/health")
async def health():

    return {
        "status": "ok",
        "model": MODEL_NAME,
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "sample_rate": SAMPLE_RATE,
    }

# ============================================================
# WEBSOCKET STT
# ============================================================

@app.websocket("/ws/stt")
async def websocket_transcribe(
    websocket: WebSocket,
):
    await websocket.accept()

    print(
        "[WS-STT] Client connected"
    )

    # --------------------------------------------------
    # Session state
    # --------------------------------------------------

    audio_buffer = bytearray()

    language = None

    try:
        # --------------------------------------------------
        # Tell client that the WebSocket is ready
        # --------------------------------------------------

        await websocket.send_json(
            {
                "type": "ready"
            }
        )

        while True:

            message = await websocket.receive()

            # ==================================================
            # BINARY AUDIO
            # ==================================================

            if message.get("bytes") is not None:

                audio_bytes = message["bytes"]

                if audio_bytes:
                    audio_buffer.extend(
                        audio_bytes
                    )

                continue

            # ==================================================
            # TEXT / JSON COMMAND
            # ==================================================

            text_message = message.get("text")

            if text_message is None:
                continue

            command = json.loads(
                text_message
            )

            command_type = command.get(
                "type"
            )

            # ==================================================
            # START
            # ==================================================

            if command_type == "start":

                language = command.get(
                    "language"
                )

                audio_buffer.clear()

                print(
                    "[WS-STT] START | "
                    f"language={language}"
                )

                continue

            # ==================================================
            # RESET
            # ==================================================

            if command_type == "reset":

                audio_buffer.clear()

                print(
                    "[WS-STT] RESET"
                )

                await websocket.send_json(
                    {
                        "type": "reset",
                        "status": "ok",
                    }
                )

                continue

            # ==================================================
            # REPLACE
            # ==================================================

            if command_type == "replace":

                audio_buffer.clear()

                print(
                    "[WS-STT] REPLACE | "
                    "buffer cleared"
                )

                continue

            # ==================================================
            # PARTIAL
            # ==================================================

            if command_type == "partial":

                total_started = (
                    time.perf_counter()
                )

                current_audio = bytes(
                    audio_buffer
                )

                print(
                    "[WS-STT] PARTIAL | "
                    f"bytes={len(current_audio)}"
                )

                result = await asyncio.to_thread(
                    transcribe_audio,
                    current_audio,
                    language,
                )

                total_seconds = (
                    time.perf_counter()
                    - total_started
                )

                await websocket.send_json(
                    {
                        "type": "transcript",
                        "status": "partial",
                        "text": result["text"],
                        "language": result["language"],
                        "language_probability": result[
                            "language_probability"
                        ],
                        "inference_seconds": result[
                            "inference_seconds"
                        ],
                        "total_seconds": total_seconds,
                    }
                )

                print(
                    "[WS-STT] PARTIAL RESULT | "
                    f"text={result['text']} | "
                    f"inference="
                    f"{result['inference_seconds']:.3f}s"
                )

                continue

            # ==================================================
            # FINAL
            # ==================================================

            if command_type == "final":

                total_started = (
                    time.perf_counter()
                )

                current_audio = bytes(
                    audio_buffer
                )

                print(
                    "[WS-STT] FINAL | "
                    f"bytes={len(current_audio)}"
                )

                result = await asyncio.to_thread(
                    transcribe_audio,
                    current_audio,
                    language,
                )

                total_seconds = (
                    time.perf_counter()
                    - total_started
                )

                await websocket.send_json(
                    {
                        "type": "transcript",
                        "status": "final",
                        "text": result["text"],
                        "language": result["language"],
                        "language_probability": result[
                            "language_probability"
                        ],
                        "inference_seconds": result[
                            "inference_seconds"
                        ],
                        "total_seconds": total_seconds,
                    }
                )

                print(
                    "[WS-STT] FINAL RESULT | "
                    f"text={result['text']} | "
                    f"inference="
                    f"{result['inference_seconds']:.3f}s"
                )

                # --------------------------------------------------
                # Start a fresh utterance after FINAL
                # --------------------------------------------------

                audio_buffer.clear()

                continue

            # ==================================================
            # UNKNOWN COMMAND
            # ==================================================

            print(
                "[WS-STT] Unknown command | "
                f"{command}"
            )

            await websocket.send_json(
                {
                    "type": "error",
                    "message": (
                        "Unknown command: "
                        f"{command_type}"
                    ),
                }
            )

    except WebSocketDisconnect:

        print(
            "[WS-STT] Client disconnected"
        )

    except Exception as e:

        print(
            "[WS-STT] WebSocket error | "
            f"{type(e).__name__}: {e}"
        )

        try:
            await websocket.close()
        except Exception:
            pass

# ============================================================
# TRANSCRIPTION HELPER
# ============================================================
def contains_arabic_script(text: str) -> bool:
    return any(
        ARABIC_UNICODE_RANGE[0] <= char <= ARABIC_UNICODE_RANGE[1]
        for char in text
    )

def run_whisper(audio, language=None):
    segments, info = whisper_model.transcribe(
        audio,
        language=language,
        beam_size=1,
        temperature=0.0,
        word_timestamps=False,
        vad_filter=False,
        condition_on_previous_text=False,
        no_speech_threshold=0.6,
        log_prob_threshold=-1.0,
        suppress_blank=True,
        compression_ratio_threshold=2.4,
        initial_prompt=(
            "This speech may contain English and Arabic, "
            "including Arabic-English mixed speech."
        ),
    )

    # Consume the generator before using the transcription.
    segments = list(segments)
    text = "".join(segment.text for segment in segments).strip()

    return text, info, segments
    
def transcribe_audio(
    audio_bytes: bytes,
    language: str | None = None,
):
    """
    Transcribe PCM16 / 16 kHz / mono audio.

    This runs Whisper inference in a worker thread so the
    WebSocket event loop remains responsive.
    """
    started = time.perf_counter()
    def result(text="", lang=None, probability=None):
        return {
            "text": text,
            "language": lang,
            "language_probability": probability,
            "inference_seconds": time.perf_counter() - started,
        }
    try:
        if not audio_bytes:
            return result(lang=language)
    
        # --------------------------------------------------
        # PCM16 bytes -> float32 [-1, 1]
        # --------------------------------------------------
    
        audio = np.frombuffer(
            audio_bytes, dtype=np.int16,
        ).astype(np.float32)
    
        audio /= 32768.0
    
        # --------------------------------------------------
        # Whisper inference
        # --------------------------------------------------
    
        text, info, segments = run_whisper(audio, language)
        
        detected_lang = info.language
        lang_prob = float(info.language_probability or 0.0)

        print(
            f"ASR | language={detected_lang} "
            f"| probability={lang_prob:.3f} | text={text}"
        )

        if not text or is_whisper_hallucination(text):
            return result(lang=detected_lang, probability=lang_prob)

        if language is None and (
            detected_lang == "ur" or lang_prob < 0.70
        ):
            ar_text, ar_info, ar_segments = run_whisper(audio, language="ar")
            print(
                f"ASR Arabic retry | text={ar_text} "
                f"| detected={ar_info.language}"
            )
            if (
                ar_text
                and contains_arabic_script(ar_text)
                and not is_whisper_hallucination(ar_text)
            ):
                text = ar_text
                detected_lang = "ar"
                lang_prob = float(ar_info.language_probability or 0.0)

        if detected_lang not in ALLOWED_LANGUAGES:
            return result(lang=detected_lang, probability=lang_prob)

        if lang_prob < 0.50:
            return result(lang=detected_lang, probability=lang_prob)

        if is_whisper_hallucination(text):
            return result(lang=detected_lang, probability=lang_prob)

        return result(text, detected_lang, lang_prob)      

    except Exception as e:
        print(
            f"[TTS_TRANSCRIBE] Error | "
            f"{type(e).__name__}: {e}"
        )
        return result(lang=language)
        
        


def is_whisper_hallucination(text: str) -> bool:
    """Returns True if the text is likely a Whisper hallucination."""
    if not text:
        return True
        
    text_clean = text.strip().lower()
    
    # 1. Check if the string is empty after removing punctuation
    text_no_punct = re.sub(r'[^\w\s]', '', text_clean)
    if not text_no_punct.strip():
        return True

    # 2. Known Whisper Hallucination phrases (add to this as you discover more)
    hallucinations = [
        "thank you",
        "thanks for watching",
        "subtitles by",
        "amara.org",
        "you",         # Very common single-word hallucination on noise
        "bye",
        "subscribe",
        "thank you very much"
    ]
    
    # If the ENTIRE transcript is just one of these phrases, it's a hallucination
    if text_no_punct.strip() in hallucinations:
        return True

    # 3. Repeating patterns (e.g., "hello hello hello hello")
    words = text_no_punct.split()
    if len(words) > 3:
        # If all words in the string are exactly the same
        if len(set(words)) == 1: 
            return True

    return False

In [ ]:
%%writefile /kaggle/working/tts_server.py

import asyncio
import json
import time
import io
import re

import numpy as np
import torch
import av


from fastapi import (
    FastAPI,
    File,
    UploadFile,
    WebSocket,
    WebSocketDisconnect,
)

from omnivoice import OmniVoice, VoiceClonePrompt

# ============================================================
# CONFIGURATION
# ============================================================

# TTS Model
MODEL_ID = "k2-fsa/OmniVoice"


print("Loading OMINI VOICE...")
tts_model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.float16
)

tts_sample_rate = 24000

print("✅ OMINI VOICE ready.")

tts_lock = asyncio.Lock()

# ============================================================
# GENERATE AUDIO CHUNKS
# ============================================================

async def generate_chunks(text: str, cancel_event: asyncio.Event):
    queue: asyncio.Queue = asyncio.Queue()
    loop = asyncio.get_running_loop()

    def worker():
        try:

            audio_tensor = tts_model.generate(
                text=text,
                normalize_text=True,
                num_step=16
            )

            if cancel_event.is_set():
                print("[TTS-WORKER] Generation cancelled mid-stream")
                return

            if isinstance(audio_tensor, torch.Tensor):
                audio_np = audio_tensor.detach().cpu().numpy()
            else:
                audio_np = np.asarray(audio_tensor, dtype=np.float32)

            audio_np = np.clip(audio_np, -1.0, 1.0)
            pcm16 = (audio_np * 32767.0).astype(np.int16)

            chunk_size = int(tts_sample_rate * 0.05)

            for i in range(0, len(pcm16), chunk_size):
                if cancel_event.is_set():
                    break
                chunk = pcm16[i:i + chunk_size]
                loop.call_soon_threadsafe(queue.put_nowait, chunk.tobytes())

        except Exception as exc:
            loop.call_soon_threadsafe(queue.put_nowait, exc)
        finally:
            loop.call_soon_threadsafe(queue.put_nowait, None)

    asyncio.create_task(asyncio.to_thread(worker))

    while True:
        if cancel_event.is_set():
            print("[TTS] Aborting chunk stream due to cancellation")
            break
            
        chunk = await queue.get()
        if chunk is None:
            break

        if isinstance(chunk, Exception):
            raise chunk

        yield chunk


# ============================================================
# FASTAPI
# ============================================================

app = FastAPI(
    title="Kaggle Faster-Whisper TTS"
)


# ============================================================
# HEALTH
# ============================================================
@app.get("/health")
async def health():

    return {
        "status": "ok",
        "model": "VoxCPM2",
        "sample_rate": tts_sample_rate,
        "format": "pcm_s16le",
    }


# ============================================================
# WEBSOCKET TTS
# ============================================================

@app.websocket("/ws/tts")
async def websocket_tts(ws: WebSocket):

    await ws.accept()
    text_queue: asyncio.Queue[str | None] = asyncio.Queue()
    cancel_event = asyncio.Event()
    
    async def receive_loop():
        """
        Continuously receive text requests from the client.
        This loop is intentionally independent from TTS generation.
        """
        try:
            while True:

                message = await ws.receive_text()
                request = json.loads(message)
                msg_type = request.get("type")

                # Handle cancellation (Barge-In)
                if msg_type == "cancel":
                    print("[WS] Received cancellation request from client")
                    cancel_event.set()

                    # Flush unserved text items from queue
                    while not text_queue.empty():
                        try:
                            text_queue.get_nowait()
                            text_queue.task_done()
                        except asyncio.QueueEmpty:
                            break
                    continue

                if msg_type != "text":
                    continue

                text = request.get("text", "").strip()
                gen_id = request.get("generation_id", 0)

                if not text:
                    await ws.send_json({
                        "type": "error",
                        "message": "Text is empty",
                    })
                    continue

                print(
                    f"[WS] Received text | "
                    f"chars={len(text)} | "
                    f"text={text}"
                )

                await text_queue.put({"text": text, "generation_id": gen_id})

        except WebSocketDisconnect:
            print("[WS] Client disconnected")

        finally:
            # Tell generation loop to stop.
            await text_queue.put(None)

    async def generation_loop():
        """
        Consume text sequentially and generate audio.
        Only ONE VoxCPM generation happens at a time.
        """
        while True:

            item = await text_queue.get()

            if item is None:
                text_queue.task_done()
                break

            text = item["text"]
            gen_id = item["generation_id"]
            
            try:

                # Clear cancellation flag for the new generation
                cancel_event.clear()

                print(f"[TTS] Starting generation | gen={gen_id} | text={text}")

                await ws.send_text(
                    json.dumps({
                        "type": "audio_start",
                        "sample_rate": tts_sample_rate,
                        "channels": 1,
                        "format": "pcm_s16le",
                        "generation_id": gen_id,
                    })
                )

                async with tts_lock:
                    async for chunk in generate_chunks(text, cancel_event):
                        if cancel_event.is_set():
                            print(f"[TTS] Interrupted generation | gen={gen_id}")
                            break
                        await ws.send_bytes(chunk)

                await ws.send_json({
                    "type": "audio_end",
                    "generation_id": gen_id,                    
                })

                print(f"[TTS] Generation completed | text={text} | gen={gen_id}")

            except WebSocketDisconnect:
                print("[TTS] Client disconnected during generation")
                break
            except Exception as exc:
                print(f"[TTS] Generation error | {type(exc).__name__}: {exc}")

                try:
                    await ws.send_json({
                        "type": "error",
                        "message": str(exc),
                    })
                except Exception:
                    break
            finally:
                text_queue.task_done()


    receive_task = asyncio.create_task(receive_loop())
    generation_task = asyncio.create_task(generation_loop())

    try:
        done, pending = await asyncio.wait(
            {
                receive_task,
                generation_task,
            },
            return_when=asyncio.FIRST_COMPLETED,
        )

        for task in pending:
            task.cancel()

        await asyncio.gather(
            *pending,
            return_exceptions=True,
        )
    except WebSocketDisconnect:
        print("[WS] WebSocket disconnected")
    except asyncio.CancelledError:
        raise
    
    except Exception as exc:
        print(f"[WS] Error | {type(exc).__name__}: {exc}")
    finally:
        receive_task.cancel()
        generation_task.cancel()
        await asyncio.gather(
            receive_task,
            generation_task,
            return_exceptions=True,
        )

        print("[WS] WebSocket session closed")

In [ ]:
import os
import subprocess
import time
import urllib.request

VENV_PATH = "/kaggle/working/venv"
PYTHON_EXECUTABLE = os.path.join(VENV_PATH, "bin", "python")
UVICORN_EXECUTABLE = os.path.join(VENV_PATH, "bin", "uvicorn")

STT_PORT = 8001
TTS_PORT = 8002

if "stt_process" not in globals():
    stt_process = None

if "tts_process" not in globals():
    tts_process = None

def stop_servers(llm_process, process_name, process_port):
    if process_name in globals() and llm_process is not None:
        if llm_process.poll() is None:
            print(f"🛑 Stopping previous {process_name} server (PID: {llm_process.pid})...")
            llm_process.terminate()
            try:
                llm_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                llm_process.kill()
                llm_process.wait()

    result = subprocess.run(["lsof", "-t", f"-i:{process_port}"], capture_output=True, text=True)
    if result.stdout.strip():
        pids = result.stdout.strip().splitlines()
        for pid in pids:
            subprocess.run(["kill", "-9", pid])
        time.sleep(1)
    print(f"✅ Port {process_port} is free.")

stop_servers(stt_process, "stt_process", STT_PORT)
stop_servers(tts_process, "tts_process", TTS_PORT)

# -------------------------------------------------------------
# Health Check Wait Function
# -------------------------------------------------------------
def wait_for_server(port, timeout=180, sleep=4):
    url = f"http://127.0.0.1:{port}/health"
    start_time = time.time()
    print(f"⏳ Waiting for model on port {port} to pass health checks...", end="")
    while time.time() - start_time < timeout:
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status == 200:
                    print(" Ready! ✅")
                    return True
                    break
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(sleep)
    print(" TIMED OUT OR CRASHED! ❌")
    return False

print("\n🚀 Starting STT and TTS Server...")

print(f"\n[1/2] Launching STT on Port {STT_PORT} (GPU 0)...")
env_gpu0 = os.environ.copy()
env_gpu0["CUDA_VISIBLE_DEVICES"] = "0"

stt_process = subprocess.Popen([
        UVICORN_EXECUTABLE,
        "stt_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        str(STT_PORT),
], env=env_gpu0, cwd="/kaggle/working", stdout=open("/tmp/stt_server.log", "w"), stderr=subprocess.STDOUT)

if not wait_for_server(STT_PORT,180,20):
    print("⚠️ Check log: /tmp/stt_server.log")

print(f"\n[2/2] Launching TTS on Port {TTS_PORT} (GPU 1)...")
env_gpu1 = os.environ.copy()
env_gpu1["CUDA_VISIBLE_DEVICES"] = "1"

tts_process = subprocess.Popen([
        UVICORN_EXECUTABLE,
        "tts_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        str(TTS_PORT),
], env=env_gpu1, cwd="/kaggle/working", stdout=open("/tmp/tts_server.log", "w"), stderr=subprocess.STDOUT)

if not wait_for_server(TTS_PORT,180,20):
    print("⚠️ Check log: /tmp/tts_server.log")

print("\n✅ Service startup sequence complete!")

In [ ]:
stop_servers(tts_process, "tts_process", TTS_PORT)
tts_process = subprocess.Popen([
        UVICORN_EXECUTABLE,
        "tts_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        str(TTS_PORT),
], env=env_gpu1, cwd="/kaggle/working", stdout=open("/tmp/tts_server.log", "w"), stderr=subprocess.STDOUT)

if not wait_for_server(TTS_PORT,180,20):
    print("⚠️ Check log: /tmp/tts_server.log")

In [ ]:
from pathlib import Path

log_path = Path("/tmp/tts_server.log")

with log_path.open("r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("\n".join(lines))

In [ ]:

print("\n==================================================")
print("🌐 STEP 3: Create NGINX FILE")
print("==================================================")

nginx_config = """
events {
    worker_connections 1024;
}

http {
    proxy_read_timeout 300s;
    proxy_connect_timeout 300s;
    proxy_send_timeout 300s;

    server {
        listen 8080;

        proxy_buffering off;
        proxy_http_version 1.1;

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;

        location /stt/ {
            proxy_pass http://127.0.0.1:8001/;

            # WebSocket support
            proxy_set_header Upgrade $http_upgrade;
            proxy_set_header Connection "upgrade";

            # Keep realtime ASR connection alive
            proxy_read_timeout 300s;
            proxy_send_timeout 300s;
        }

        location /tts/ {
            proxy_pass http://127.0.0.1:8002/;

            # WebSocket support
            proxy_set_header Upgrade $http_upgrade;
            proxy_set_header Connection "upgrade";

            # Keep realtime ASR connection alive
            proxy_read_timeout 300s;
            proxy_send_timeout 300s;
        }
    }
}
"""

with open("/tmp/nginx.conf", "w") as f:
    f.write(nginx_config)

if "nginx_process" not in globals():
    nginx_process = None

def stop_nginx_servers(_process, _process_name, _process_port):
    if _process_name in globals() and _process is not None:
        if _process.poll() is None:
            print(f"🛑 Stopping previous {_process_name} server (PID: {_process.pid})...")
            _process.terminate()
            try:
                _process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                _process.kill()
                _process.wait()

    result = subprocess.run(["lsof", "-t", f"-i:{_process_port}"], capture_output=True, text=True)
    if result.stdout.strip():
        pids = result.stdout.strip().splitlines()
        for pid in pids:
            subprocess.run(["kill", "-9", pid])
        time.sleep(2)
    print(f"✅ Port {_process_port} is free.")

stop_nginx_servers(nginx_process, "nginx_process", 8080)

nginx_process = subprocess.Popen([
    "nginx",
    "-c", "/tmp/nginx.conf",
    "-g", "daemon off;"
])

print("✅ NGINX running on port 8080")

In [ ]:
import re

print("\n==================================================")
print("🌐 STEP 4: EXPOSING PORT 8080 VIA CLOUDFLARE")
print("==================================================")

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)



print("\nStarting tunnel...")
for line in tunnel_process.stdout:
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n🚀 SUCCESS! Your public Base URL is:")
            print(match.group(0))
            break